In [1]:
import ee 
import geemap
import geopandas as gpd

import pprint as pp

ykf = gpd.read_file('./data/YKflats_roi_shape.shp')

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

Collection of Sentinel-2 Images on target date

In [3]:

roi_ee = ee.Geometry(ykf.iloc[0].geometry.__geo_interface__)

date = '2020-05-29'
date_plus1d = '2020-05-30' 
#Required to filter by date, but end_date is exclusive so not in image collection

s2_spec_reflec = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi_ee)
    .filterDate(date, date_plus1d)
)


# pp.pp(s2_spec_reflec.first().getInfo())
s2_total_imgs = s2_spec_reflec.size().getInfo()

s2_spec_reflec = s2_spec_reflec.map(lambda img: img.clip(roi_ee))
s2_spec_reflec = s2_spec_reflec.mosaic()


In [4]:
s2_cl_prob = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
    .filterBounds(roi_ee)
    .filterDate(date, date_plus1d)
)
s2_cl_prob = s2_cl_prob.map(lambda img: img.clip(roi_ee))
s2_cl_prob = s2_cl_prob.mosaic()

s2_cl_mask = s2_cl_prob.select('probability').lt(25).rename('cl_binary')

In [6]:
# Get the boundary of the Sentinel-2 clipped image
s2_data_mask = s2_spec_reflec.mask().reduce(ee.Reducer.anyNonZero())
s2_boundary = s2_data_mask.reduceToVectors(
    geometry=roi_ee,
    geometryType='polygon',
    scale=10,
    maxPixels=1e13
)

s2_boundary = ee.Feature(s2_boundary.toList(s2_boundary.size()).get(0))


Collection of Landsat Images on target date

In [7]:
def optical_rescale(img):
    "Only converts optical bands, thermal bands not included"
    img = img.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    )
    img = img.multiply(0.0000275).add(-0.2)

    return img


ls_spec_reflec = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(roi_ee)
                  .filterDate(date, date_plus1d)
)

pp.pp(ls_spec_reflec.getInfo())

ls_spec_reflec = ls_spec_reflec.map(lambda img: img.clip(roi_ee))

ls_qa = ls_spec_reflec.select('QA_PIXEL').mosaic()


ls_spec_reflec = ls_spec_reflec.map(optical_rescale)

ls_total_imgs = ls_spec_reflec.size().getInfo()
print(ls_total_imgs)

ls_spec_reflec = ls_spec_reflec.mosaic()


{'type': 'ImageCollection',
 'bands': [],
 'version': 1729676042466268,
 'id': 'LANDSAT/LC08/C02/T1_L2',
 'properties': {'type_name': 'ImageCollection',
                'keywords': ['cfmask',
                             'cloud',
                             'fmask',
                             'global',
                             'l8sr',
                             'landsat',
                             'lasrc',
                             'lst',
                             'reflectance',
                             'sr',
                             'usgs'],
                'visualization_1_bands': 'SR_B5,SR_B4,SR_B3',
                'thumb': 'https://mw1.google.com/ges/dd/images/LANDSAT_SR_thumb.png',
                'visualization_1_max': '30000.0',
                'description': '<p>This dataset contains atmospherically '
                               'corrected\n'
                               'surface reflectance and land surface '
                               'temp

In [8]:
ls_full_mask = ls_qa.bitwiseAnd(1 << 8).eq(0)
ls_shaddow_mask = ls_qa.bitwiseAnd(4).eq(0)
ls_cloud_mask = ls_qa.bitwiseAnd(2).eq(0)
ls_ddv_mask = ls_qa.bitwiseAnd(1).eq(0)


In [9]:
# Get the bounds of the Landsat image

ls_data_mask = ls_spec_reflec.mask().reduce(ee.Reducer.anyNonZero())
ls_boundary = ls_data_mask.reduceToVectors(
    geometry=roi_ee,
    geometryType='polygon',
    scale=10,
    maxPixels=1e13
)

ls_boundary = ee.Feature(ls_boundary.toList(ls_boundary.size()).get(2))

In [10]:
Map = geemap.Map()

s2_true_col_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
}

s2_cloud_prob_params = {
    'bands': ['probability'],
    'min': 0,
    'max': 100
}

ls_true_col_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0,
    'max': 0.3
}


Map.addLayer(s2_spec_reflec, s2_true_col_params, f'S2 True Color Composite on {date}')
Map.addLayer(ls_spec_reflec, ls_true_col_params, f'LS8 True Color Composite on {date}')

#Map.addLayer(s2_cl_prob, s2_cloud_prob_params, 'S2 Cloud Probability')
Map.addLayer(s2_cl_mask, {'min': 0, 'max': 1}, 'S2 Cloud Mask')
Map.addLayer(ls_full_mask, {'min': 0, 'max': 1}, 'LS Full QA Mask')
Map.addLayer(ls_cloud_mask, {'min': 0, 'max': 1}, 'LS Cloud Mask')
Map.addLayer(ls_shaddow_mask, {'min': 0, 'max': 1}, 'LS Shaddow Mask')

# Map.addLayer(s2_boundary, {'color': 'blue'}, 'S2 Boundary')
# Map.addLayer(ls_boundary, {'color': 'green'}, 'LS Boundary')
# Map.addLayer(roi_ee, {'color': 'red'}, 'ROI Outline')
Map.centerObject(roi_ee, zoom=10)
Map


Map(center=[66.51859072213907, -146.00018537875346], controls=(WidgetControl(options=['position', 'transparent…

Export data masks and images